# Coswara Respiratory Sound EDA

This notebook explores the local Coswara dataset, checks audio inventory quality, and reviews class balance before model training.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torchaudio

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from respiratory_ai.config import DEFAULT_SOUND_TYPES
from respiratory_ai.data.cleaning import clean_audio_manifest, list_audio_files
from respiratory_ai.data.dataset import CoswaraDataset
from respiratory_ai.eda.analysis import plot_class_distribution


In [2]:
DATA_DIR = (PROJECT_ROOT / "coswara-data").resolve()

if not list_audio_files(DATA_DIR):
    extracted_dir = DATA_DIR / "Extracted_data"
    if extracted_dir.exists():
        DATA_DIR = extracted_dir
    else:
        print(f"No extracted audio files found in {DATA_DIR}.")
        print("The Coswara archive is still compressed as .tar.gz.* files.")
        print("Extract it first, then rerun this cell:")
        print(f'  cd "{DATA_DIR}"')
        print("  python extract_data.py")
        print("Or set DATA_DIR to the extracted folder once it exists.")

audio_files = list_audio_files(DATA_DIR)
print(f"Audio files found: {len(audio_files)}")
len(audio_files)

Audio files found: 13376


13376

In [3]:
manifest = clean_audio_manifest(DATA_DIR, out_path="../artifacts/audio_manifest.csv")
manifest.head()

,file_name,relative_path,suffix,size_bytes,is_valid,record_count
0,breathing-deep.wav,20200413\20200413\0Rlzhiz6bybk77wdLjxwy7yLDhg1...,.wav,2400300,True,1
1,breathing-shallow.wav,20200413\20200413\0Rlzhiz6bybk77wdLjxwy7yLDhg1...,.wav,1400876,True,1
2,cough-heavy.wav,20200413\20200413\0Rlzhiz6bybk77wdLjxwy7yLDhg1...,.wav,499756,True,1
3,cough-shallow.wav,20200413\20200413\0Rlzhiz6bybk77wdLjxwy7yLDhg1...,.wav,466988,True,1
4,counting-fast.wav,20200413\20200413\0Rlzhiz6bybk77wdLjxwy7yLDhg1...,.wav,565292,True,1


In [4]:
manifest["suffix"].value_counts()

suffix
.wav    13376
Name: count, dtype: int64

In [5]:
dataset = CoswaraDataset(
    data_dir=DATA_DIR,
    sound_types=DEFAULT_SOUND_TYPES,
    sr=16000,
    n_mels=64,
    max_duration=5.0,
)

labels = [dataset[i][1] for i in range(len(dataset))]
class_names = dataset.class_names
class_names

['breathing_deep',
 'breathing_shallow',
 'cough_heavy',
 'cough_shallow',
 'vowel_a',
 'vowel_i',
 'vowel_o',
 'counting_normal',
 'counting_fast',
 'speech',
 'voice']

In [6]:
label_names = [class_names[label] for label in labels]
label_df = pd.Series(label_names).value_counts().sort_index().rename_axis("class").reset_index(name="count")
label_df

,class,count
0,breathing_deep,1461
1,cough_heavy,1460
2,cough_shallow,1457
3,counting_fast,1458
4,counting_normal,1459
5,vowel_a,1458
6,vowel_o,1454


In [7]:
plot_class_distribution(label_names, "../artifacts/class_distribution.png")
plt.show()

In [8]:
sample_waveform = []
for i in range(min(3, len(dataset))):
    sample, _ = dataset[i]
    sample_waveform.append(sample.squeeze().numpy())

sample_waveform[0].shape

(64, 157)

In [9]:
sample_mel, sample_label = dataset[0]
print(sample_mel.shape)
print(sample_label)

torch.Size([1, 64, 157])
0


## Practical notes

- The dataset is normalized to 16 kHz and 5 seconds.
- Each sample returns a log Mel-spectrogram and label.
- Class balance should be inspected before training to avoid skewed learning.
- This notebook is for analysis and model preparation; training is still done via the project pipeline.